#  Employee Attrition Classification

A formal **Machine Learning classification project** that predicts whether
an employee is likely to leave an organization.

## Business Objective
Use employee information such as income, job level, overtime and
satisfaction to classify employees into:

- **Stay**
- **Leave**

## Workflow
**Business Data → Explore → Features & Target → Train/Test Split → Preprocessing → Classification Model → Evaluation**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Import the Dataset

The dataset contains employee-related information and the target column
`Attrition`, which indicates whether an employee left the organization.

In [ ]:
employee_data = pd.read_csv("data/employee-attrition-classification.csv")

print("Dataset shape:", employee_data.shape)
print("\nColumns:")
print(employee_data.columns.tolist())

display(employee_data.head())

Dataset shape: (24, 11)

Columns:
['Age', 'MonthlyIncome', 'JobLevel', 'YearsAtCompany', 'JobSatisfaction', 'OverTime', 'WorkLifeBalance', 'JobInvolvement', 'NumCompaniesWorked', 'TotalWorkingYears', 'Attrition']


   Age  MonthlyIncome  JobLevel  YearsAtCompany  JobSatisfaction OverTime  WorkLifeBalance  JobInvolvement  NumCompaniesWorked  TotalWorkingYears Attrition
0   24           2800         1               1                2      Yes                2               2                   3                  2       Yes
1   29           4200         2               3                3       No                3               3                   2                  6        No
2   31           5100         2               5                4       No                3               4                   1                  8        No
3   35           6200         3               7                3      Yes                2               3                   2                 11       Yes
4   41           8500         4              12                4       No                4               4                   1                 18        No

## 2. Understand the Target

`Attrition` is the target variable.

This is a **binary classification** problem because there are two possible
outcomes: `Yes` and `No`.

In [ ]:
print("Attrition distribution:")
print(employee_data["Attrition"].value_counts())

print("\nAttrition percentage:")
print((employee_data["Attrition"].value_counts(normalize=True) * 100).round(2))

Attrition distribution:
Attrition
No     14
Yes    10

Attrition percentage:
Attrition
No     58.33
Yes    41.67


## 3. Separate Features and Target

In [ ]:
X = employee_data.drop("Attrition", axis=1)
y = employee_data["Attrition"]

print("Feature columns:", list(X.columns))
print("Target column:", y.name)

Feature columns: ['Age', 'MonthlyIncome', 'JobLevel', 'YearsAtCompany', 'JobSatisfaction', 'OverTime', 'WorkLifeBalance', 'JobInvolvement', 'NumCompaniesWorked', 'TotalWorkingYears']
Target column: Attrition


## 4. Train/Test Split

The test set is kept separate so that the final model evaluation is
performed on data it did not train on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 18
Testing samples: 6


## 5. Preprocessing

The dataset contains:

- **Numerical features** → median imputation
- **Categorical feature (`OverTime`)** → most-frequent imputation + One-Hot Encoding

A `ColumnTransformer` keeps these transformations organized.

In [ ]:
categorical_features = ["OverTime"]
numeric_features = [column for column in X.columns if column not in categorical_features]

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['Age', 'MonthlyIncome', 'JobLevel', 'YearsAtCompany', 'JobSatisfaction', 'WorkLifeBalance', 'JobInvolvement', 'NumCompaniesWorked', 'TotalWorkingYears']
Categorical features: ['OverTime']


## 6. Train the Classification Model

A Random Forest Classifier is placed inside a complete Pipeline so that
preprocessing and modelling are handled together.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Classification model trained successfully!")

Classification model trained successfully!


## 7. Make Predictions

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

display(results)

Actual Predicted
   Yes       Yes
    No        No
    No        No
    No        No
    No        No
   Yes       Yes

## 8. Classification Metrics

We evaluate the model using:

- **Accuracy**
- **Precision**
- **Recall**
- **F1 Score**

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="Yes", zero_division=0)
recall = recall_score(y_test, y_pred, pos_label="Yes", zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label="Yes", zero_division=0)

print("Model Performance")
print("-----------------")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")

Model Performance
-----------------
Accuracy : 1.000
Precision: 1.000
Recall   : 1.000
F1 Score : 1.000


## 9. Confusion Matrix

The confusion matrix shows correct and incorrect predictions for employees
who stayed and employees who left.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])

print("Confusion Matrix")
print("----------------")
print(cm)

print("\nRows = Actual [Stay, Leave]")
print("Columns = Predicted [Stay, Leave]")

Confusion Matrix
----------------
[[4 0]
 [0 2]]

Rows = Actual [Stay, Leave]
Columns = Predicted [Stay, Leave]


## 10. Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=["No", "Yes"],
        target_names=["Stay", "Leave"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

        Stay       1.00      1.00      1.00         4
       Leave       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


##  Business Interpretation

The model demonstrates how Machine Learning can be applied to a practical
HR problem: identifying employees who may be at higher risk of attrition.

In a real organization, the model could support:
- employee retention analysis
- workforce planning
- HR decision support
- identifying factors associated with employee turnover

> **Important:** A predictive model should support—not replace—human
> decision-making in employment contexts.

##  Key Takeaways

- Classification predicts discrete categories.
- `Attrition` is the target variable.
- Categorical data can be converted using `OneHotEncoder`.
- `ColumnTransformer` handles different feature types.
- `Pipeline` keeps preprocessing and modelling together.
- Accuracy alone is not enough; precision, recall and F1 are also useful.
- Confusion matrices show class-level prediction performance.

### Next Step
**Model Evaluation & Improvement** — cross-validation, hyperparameter
tuning and comparing multiple classification models.